In [1]:
from utils.data import load


data = load(f'/home/burger/bioinfo/project/data/pdb.hdf5')

In [ ]:
print('seq:', data[0].shape)
print('node_pos:', data[1].shape)
print('node_idx:', data[2].shape)
print('edge_nho:', data[3].shape)
print('edge_idx:', data[4].shape)
print('lab:', data[5].shape)

seq: (26041138,)
node_pos: (26041138, 5, 3)
node_idx: (106974,)
edge_nho: (2, 18611806)
edge_idx: (106974,)
lab: (106973,)


In [3]:
from torch_geometric.data import Data
import torch as pt
from torch import from_numpy


seqs, node_pos, node_idx, edge_nho, edge_idx, lab = data
seq_list, graph_list, lab_list = [], [], []

for i in range(len(lab)):
    seq = from_numpy(seqs[node_idx[i]: node_idx[i+1]])
    len_seq = len(seq)
    # shape:(len_seq,5,3) 0:N, 1:α, 2:C, 3:O, 4:β 
    pos = from_numpy(node_pos[node_idx[i]: node_idx[i+1]])
    # N-α-β这个夹角反映了氨基酸的空间结构，在化学键确定的前提下，N和β之间的距离就能反应角度 
    node_attr = pt.norm(pos[:, 0] - pos[:, 4], dim=1, keepdim=True)
    # 连接关系(肽键), 单向
    tai = pt.stack((pt.arange(0, len_seq-1), pt.arange(1, len_seq)), dim=0)
    # 连接关系(氢键), 单向
    nho = pt.stack((from_numpy(edge_nho[0][edge_idx[i] : edge_idx[i+1]]),
                    from_numpy(edge_nho[1][edge_idx[i] : edge_idx[i+1]])), dim=0)
    edge = pt.cat((tai, tai.flip(0), nho, nho.flip(0)), dim=1)
    # 边长 
    # 肽键：羧基碳接氨基氮
    tai_len = pt.norm(pos[tai[0], 2] - pos[tai[1], 0], dim=1)
    nho_len = pt.norm(pos[nho[0], 0] - pos[nho[1], 3], dim=1)
    edge_len = pt.cat((tai_len, tai_len, nho_len, nho_len), dim=0)
    # edge_attr : 键长, is_peptide, direction, is_hbond, 肽键：±1 氢键：0
    is_peptide = pt.cat((pt.ones(tai_len.size(0)*2), pt.zeros(nho_len.size(0)*2)), dim=0)
    is_hbond = 1 - is_peptide
    direction = pt.cat((pt.ones(tai_len.size(0)), -pt.ones(tai_len.size(0)), pt.zeros(nho_len.size(0)*2)), dim=0)
    edge_attr = pt.stack((edge_len, is_peptide, direction, is_hbond), dim=1) # [num_edge, num_edge_feature]
    node2seq = pt.arange(len_seq, dtype=pt.long)
    graph = Data(x=node_attr, edge_index=edge, edge_attr=edge_attr, node2seq=node2seq)
    seq_list.append(seq)
    graph_list.append(graph)
    lab_list.append(lab[i])
pt.save((seq_list, graph_list, lab_list), f'./data/engineered_data.pt')

In [4]:
a = pt.tensor([1,2,3])
b = pt.tensor([4,5,6])
c = pt.cat((a,b), dim=0)
print(c)

tensor([1, 2, 3, 4, 5, 6])


In [5]:
d = pt.stack((a,b), dim=0)
print(d)

tensor([[1, 2, 3],
        [4, 5, 6]])


In [6]:
e = pt.cat([d ,d], dim=1)
print(e)

tensor([[1, 2, 3, 1, 2, 3],
        [4, 5, 6, 4, 5, 6]])
